## Journey Agent

Creating structure format for input and output\
Prompt and LLM chain for generating a storyboard based off the input\
Test with sample inputs

Input a description of what the user journey storyboard should be. Four input options: Persona, Goal, Product, Scenario. The Agent should generate an output of a list of panels, each containing the panel number, action, context, and emotion.

### Set Up

In [ ]:
# Core imports
import json
import os
from typing import List, Dict, Optional, Literal

# LangChain imports
from langchain_ollama import ChatOllama
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotPromptTemplate,
    PromptTemplate,
    MessagesPlaceholder
)
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableBranch
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Pydantic for structured output
from pydantic import BaseModel, Field


In [2]:
# Configuration
USE_REMOTE = False
#REMOTE_URL = 
#LOCAL_URL = "http://localhost:11434"
#OLLAMA_BASE_URL = REMOTE_URL if USE_REMOTE else LOCAL_URL

# Import utilities
import sys
sys.path.insert(0, '../inclass')

from llm_utils import get_llm, get_chat_model

# Get models - use qwen3:4b for reasoning tasks
model = get_llm(use_remote=USE_REMOTE, model="qwen3:4b")
chat_model = get_chat_model(use_remote=USE_REMOTE, model="qwen3:4b")

Using Ollama at http://localhost:11434
Model: qwen3:4b
Using Ollama at http://localhost:11434
Model: qwen3:4b


In [3]:
# initialize LLM
llm = ChatOllama(
    model="qwen-3:4b",
    temperature=0.7,
    #base_url=OLLAMA_BASE_URL
)

In [4]:
# load in files
with open('sample_inputs.json', "r") as f:
    SAMPLE_INPUTS = json.load(f)


### Pydantic Objects (Data Models)

In [5]:
# create input schema for Journey Agent
class StoryboardInput(BaseModel):
    """Input schema for the StoryboardAgent."""
    persona: str = Field(..., description = "The user's persona or role, " \
    "which may influence their goals and constraints.")
    goal: str = Field(..., description = "The user's high-level goal or task.")
    product: str = Field(..., description = "The product or service the user is " \
    "interacting with, which may have specific features or limitations.")
    scenario: str = Field(..., description = "The specific context or situation "
    "in which the user is trying to achieve their goal, which may include " \
    "environmental factors or constraints.")

In [6]:
# create output panel for Journey Agent
class Panel(BaseModel):
    """Output panel for the StoryboardAgent."""
    panel_number: int = Field(..., description = "The sequential number of the " \
    "panel in the storyboard.")
    action: str = Field(..., description = "A concise description of the user's " \
    "action or interaction in this panel.")
    context: str = Field(..., description = "Additional context or details about " \
    "the user's state or environment during this panel.")
    emotion: str = Field(..., description = "A one or two word description of the " \
    "user's emotional state during this panel (e.g., 'frustrated', 'satisfied', 'confused').")

In [7]:
# class output schema for Journey Agent
class StoryboardOutput(BaseModel):
    """Output schema for the StoryboardAgent."""
    panels: List[Panel] = Field(..., description = "A list of panels that make up the storyboard, " \
    "each describing a step in the user's journey towards their goal.")

### Journey Parser

In [ ]:
# journey_parser = PydanticOutputParser(pydantic_object=StoryboardOutput)
# # create prompt template for Journey Agent
# journey_prompt = ChatPromptTemplate.from_messages([
#     #input_variables=["persona", "goal", "product", "scenario"],
#     ("system", "You are a user experience designer creating a storyboard to illustrate a user's journey with a product."),
#     ("human", """"Given the following user information, create a storyboard consisting of 4 - 6 panels that describes the user's action, context, and emotion. Be concise and focus on the key interaction in context of the product and the user's goals.\n\n" \
#         "User Persona: {persona}\n" 
#         "User Goal: {goal}\n" 
#         "Product: {product}\n"
#         "Scenario: {scenario}\n\n" 
#         "Format your response as JSON with this structure:
#         {{
#         "panels": [
#             {{
#             "panel_number": The sequential number of this panel in the storyboard,
#             "action": A concise description of the user's action or interaction in this panel,
#             "context": Additional context or details about the user's state or environment during this panel.,
#             "emotion": A one or two word description of the user's emotional state during this panel (e.g., 'frustrated', 'satisfied', 'confused').
#             }}
#         ]
#         }}
#         Return only valid JSON, no extra text.""")
#     ]
# )
# journey_chain = journey_prompt | chat_model | journey_parser

In [ ]:
journey_parser = PydanticOutputParser(pydantic_object=StoryboardOutput)
# create prompt template for Journey Agent
journey_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a user experience designer creating a storyboard to illustrate a user's journey with a product."),
    ("human", """Given the following user information, create a storyboard consisting of 6 panels \
that describes the user's action, context, and emotion. Be concise and focus on the key interaction \
in context of the product and the user's goals.

User Persona: {persona}
User Goal: {goal}
Product: {product}
Scenario: {scenario}

Format your response as JSON with this structure:
{{
  "panels": [
    {{
      "panel_number": The sequential number of this panel in the storyboard,
      "action": "concise description of user action",
      "context": "details about user state or environment",
      "emotion": "one or two word emotion"
    }}
  ]
}}

Return only valid JSON, no extra text.""")
])

journey_chain = journey_prompt | chat_model | journey_parser

In [12]:
# test the Journey Agent on a sample input
test_input = SAMPLE_INPUTS[0]
result = journey_chain.invoke({"persona": test_input["persona"], 
                               "goal": test_input["goal"], 
                               "product": test_input["product"], 
                               "scenario": test_input["scenario"]}
)

print("Sample Output:")
#print(result)
for p in result.panels:
    print(f"Panel Number: {p.panel_number}")
    print(f"Action: {p.action}")
    print(f"Context: {p.context}")
    print(f"Emotion: {p.emotion}")
    print("-" * 40)

Sample Output:
Panel Number: 1
Action: Checks phone to confirm 10 minutes until class
Context: Dorm room at 8:50 PM, phone screen shows class time 9:00 PM
Emotion: Urgent
----------------------------------------
Panel Number: 2
Action: Opens food delivery app
Context: Phone screen displays campus location pin on map
Emotion: Anxious
----------------------------------------
Panel Number: 3
Action: Filters for meals under $5 and 5-minute delivery
Context: App shows campus map with price and time filters active
Emotion: Hopeful
----------------------------------------
Panel Number: 4
Action: Selects and confirms $4.50 meal with 5-minute delivery
Context: App displays order confirmation with estimated delivery time
Emotion: Relieved
----------------------------------------


### Inputing storyboard prompt

In [16]:
# function to collect user input for the Journey Agent
def collect_input() -> StoryboardInput:
    """Collect user input for the storyboard agent."""
    persona = input("Enter the user persona: ").strip()
    goal = input("Enter the user goal: ").strip()
    product = input("Enter the product: ").strip()
    scenario = input("Enter the scenario: ").strip()
    return StoryboardInput(persona=persona, goal=goal, product=product, scenario=scenario)

In [17]:
# Test the Journey Agent with user input
user_input = collect_input()
print("Input collected:")
print(f"Persona: {user_input.persona}")
print(f"Goal: {user_input.goal}")
print(f"Product: {user_input.product}")
print(f"Scenario: {user_input.scenario}")
print("=" * 40)
print("Generating storyboard...")
print("=" * 40)
result = journey_chain.invoke({"persona": user_input.persona, 
                                "goal": user_input.goal, 
                                "product": user_input.product, 
                                "scenario": user_input.scenario})
print("Sample Output:")
for p in result.panels:
    print(f"Panel Number: {p.panel_number}")
    print(f"Action: {p.action}")
    print(f"Context: {p.context}")
    print(f"Emotion: {p.emotion}")
    print("-" * 40)

Input collected:
Persona: Working professional
Goal: book a last minute flight for a business trip
Product: travel booking website
Scenario: 2 hours before departure
Generating storyboard...
Sample Output:
Panel Number: 1
Action: Checks digital calendar to confirm business trip timing
Context: Office desk at 2:00 PM, 2 hours before flight departure
Emotion: Anxious
----------------------------------------
Panel Number: 2
Action: Opens travel booking website on laptop
Context: Browser window with website homepage
Emotion: Determined
----------------------------------------
Panel Number: 3
Action: Searches for flights departing within next 2 hours
Context: Real-time search results showing last-minute options
Emotion: Stressed
----------------------------------------
Panel Number: 4
Action: Completes booking with credit card payment
Context: Confirmation screen with flight details
Emotion: Relieved
----------------------------------------
